# Fase 0 — Modelo Antispam en Tiempo Real

## Por qué se rediseñaron las variables

El modelo original (Colab) usaba `duracion`, `frecuencia` y `prefijo`. El problema: **la duración de la llamada solo se conoce después de que terminó**, por lo que no puede usarse para decidir si bloquear la llamada *antes* de que suene el teléfono (requisito RNF-02: latencia < 200ms, y RF-04: bloqueo automático).

Este notebook entrena el modelo con 6 variables que **sí se conocen en el instante en que llega la llamada entrante**:

1. `en_contactos` (0/1) — si está guardado, nunca se bloquea (se deja sonar).
2. `prefijo_riesgo` (0/1) — si el código de país del número es infrecuente para el usuario (no coincide con países de contactos/familiares habituales).
3. `frecuencia_llamadas_numero` — cuántas veces llamó ese número en los últimos días.
4. `hora_del_dia` (0-23) — hora de la llamada entrante.
5. `es_numero_corto_o_extraño` (0/1) — formato atípico (VoIP/spoofing).
6. `reportes_previos` — veces que la comunidad marcó ese número como spam.

`duracion` se descarta como variable de predicción en tiempo real, pero se reutiliza en el feedback loop (RF-05): si una llamada de un número desconocido duró muy poco, refuerza el patrón de spam para la próxima vez que ese número llame.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

np.random.seed(42)

## 1. Generar dataset simulado

Se simula primero una clase real oculta `spam` (0/1) y luego, condicionado a esa clase, se generan las 6 variables **con ruido**, para que el problema no sea trivialmente separable (una exactitud de 1.00 en un dataset simulado suele ser señal de fuga de datos o de una regla determinista, no de un modelo realista).

In [ ]:
cantidad = 2000

# Prevalencia real de spam entre llamadas de números desconocidos
spam_real = np.random.choice([0, 1], size=cantidad, p=[0.65, 0.35])

en_contactos = np.empty(cantidad, dtype=int)
prefijo_riesgo = np.empty(cantidad, dtype=int)
frecuencia_llamadas_numero = np.empty(cantidad, dtype=int)
hora_del_dia = np.empty(cantidad, dtype=int)
es_numero_corto_o_extrano = np.empty(cantidad, dtype=int)
reportes_previos = np.empty(cantidad, dtype=int)

for i in range(cantidad):
    if spam_real[i] == 1:
        # Spammers casi nunca están guardados como contacto
        en_contactos[i] = np.random.choice([0, 1], p=[0.98, 0.02])
        # Suelen usar prefijos poco habituales para el usuario (o spoofing)
        prefijo_riesgo[i] = np.random.choice([0, 1], p=[0.30, 0.70])
        # Los bots llaman muchas veces en poco tiempo
        frecuencia_llamadas_numero[i] = np.random.poisson(lam=14) + 1
        # Call centers concentran llamadas en horario laboral
        hora_del_dia[i] = int(np.clip(np.random.normal(loc=13, scale=3.5), 0, 23))
        # Formato de número atípico (VoIP/spoofing) más común en spam
        es_numero_corto_o_extrano[i] = np.random.choice([0, 1], p=[0.65, 0.35])
        # Más reportes previos de la comunidad
        reportes_previos[i] = np.random.poisson(lam=6)
    else:
        # Llamada legítima: puede o no estar guardada como contacto
        en_contactos[i] = np.random.choice([0, 1], p=[0.45, 0.55])
        # Prefijo casi siempre familiar/local (con alguna excepción real, ej. familiar en el extranjero)
        prefijo_riesgo[i] = np.random.choice([0, 1], p=[0.88, 0.12])
        # Llama pocas veces
        frecuencia_llamadas_numero[i] = np.random.poisson(lam=2) + 1
        # Puede llamar a cualquier hora (incluye husos horarios distintos si es familiar en el extranjero)
        hora_del_dia[i] = np.random.randint(0, 24)
        # Rara vez tiene formato atípico
        es_numero_corto_o_extrano[i] = np.random.choice([0, 1], p=[0.97, 0.03])
        # Casi nunca reportado por la comunidad
        reportes_previos[i] = np.random.poisson(lam=0.15)

df = pd.DataFrame({
    'en_contactos': en_contactos,
    'prefijo_riesgo': prefijo_riesgo,
    'frecuencia_llamadas_numero': frecuencia_llamadas_numero,
    'hora_del_dia': hora_del_dia,
    'es_numero_corto_o_extrano': es_numero_corto_o_extrano,
    'reportes_previos': reportes_previos,
    'spam': spam_real
})

# Ruido de etiqueta: ~3% de casos reales son ambiguos/mal etiquetados (ej. errores de reporte comunitario)
n_ruido = int(cantidad * 0.03)
idx_ruido = np.random.choice(df.index, size=n_ruido, replace=False)
df.loc[idx_ruido, 'spam'] = 1 - df.loc[idx_ruido, 'spam']

print("DATASET SIMULADO")
print(df.head(10))
print("\nDistribución de la clase 'spam':")
print(df['spam'].value_counts(normalize=True))

## 2. Análisis exploratorio rápido

In [ ]:
print(df.describe())
print("\nPromedio de cada variable por clase:")
print(df.groupby('spam').mean().T)

## 3. Dividir dataset (train / test)

In [ ]:
variables = [
    'en_contactos',
    'prefijo_riesgo',
    'frecuencia_llamadas_numero',
    'hora_del_dia',
    'es_numero_corto_o_extrano',
    'reportes_previos'
]

X = df[variables]
y = df['spam']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", len(X_train))
print("Test:", len(X_test))

## 4. Entrenar RandomForestClassifier

In [ ]:
modelo = RandomForestClassifier(
    n_estimators=100,
    max_depth=6,
    random_state=42
)

modelo.fit(X_train, y_train)

## 5. Métricas de evaluación (RNF-01: precisión mínima 90%)

In [ ]:
predicciones = modelo.predict(X_test)
probabilidades = modelo.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, predicciones)
precision = precision_score(y_test, predicciones)
recall = recall_score(y_test, predicciones)
f1 = f1_score(y_test, predicciones)

print("MÉTRICAS DEL MODELO")
print("-" * 40)
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

print("\nREPORTE DE CLASIFICACIÓN")
print(classification_report(y_test, predicciones, target_names=["No Spam", "Spam"]))

## 6. Matriz de confusión

In [ ]:
cm = confusion_matrix(y_test, predicciones)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Spam", "Spam"])
disp.plot(cmap='Blues')
plt.title("Matriz de Confusión")
plt.show()

## 7. Importancia de variables

Esto ayuda a priorizar qué variables deben pesar más en la lógica de reglas que se implementará en Kotlin (RF simple → if/else).

In [ ]:
importancias = modelo.feature_importances_

orden = np.argsort(importancias)[::-1]
variables_ordenadas = [variables[i] for i in orden]
importancias_ordenadas = importancias[orden]

plt.figure(figsize=(9, 4))
plt.bar(variables_ordenadas, importancias_ordenadas, color='#4C72B0')
plt.title("Importancia de Variables")
plt.ylabel("Importancia")
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print("IMPORTANCIA POR VARIABLE")
for v, i in zip(variables_ordenadas, importancias_ordenadas):
    print(f"  {v}: {i:.4f}")

## 8. Validación cruzada (K=5)

In [ ]:
modelo_cv = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
scores = cross_val_score(modelo_cv, X, y, cv=5, scoring='accuracy')

print("VALIDACIÓN CRUZADA (K=5)")
print("-" * 40)
for i, s in enumerate(scores):
    print(f"  Fold {i+1}: {s:.4f}")
print(f"\nPromedio Accuracy : {scores.mean():.4f}")
print(f"Desviación Estándar: {scores.std():.4f}")

plt.figure(figsize=(7, 4))
plt.bar([f"Fold {i+1}" for i in range(5)], scores, color='#4C72B0')
plt.axhline(scores.mean(), color='red', linestyle='--', label=f"Promedio: {scores.mean():.2f}")
plt.title("Accuracy por Fold - Validación Cruzada")
plt.ylabel("Accuracy")
plt.ylim(0.7, 1.02)
plt.legend()
plt.tight_layout()
plt.show()

## 9. Predicción con casos nuevos (los 3 escenarios del proyecto)

Se aplica el umbral de RF-04: si `Prob. Spam > 0.85` → bloqueo/desvío automático.

In [ ]:
UMBRAL_BLOQUEO = 0.85

casos_nuevos = pd.DataFrame({
    'en_contactos':                [1,   0,   0],
    'prefijo_riesgo':               [0,   1,   1],
    'frecuencia_llamadas_numero':   [3,   1,   22],
    'hora_del_dia':                 [20,  9,   11],
    'es_numero_corto_o_extrano':    [0,   0,   1],
    'reportes_previos':             [0,   0,   9],
})

descripciones = [
    "Contacto guardado (familiar)",
    "Desconocido probablemente legítimo (ej. familiar en el extranjero)",
    "Desconocido probablemente spam/robocall"
]

resultado = modelo.predict(casos_nuevos)
probabilidad = modelo.predict_proba(casos_nuevos)

print("PREDICCIÓN CON CASOS NUEVOS")
print("-" * 70)
for desc, r, p in zip(descripciones, resultado, probabilidad):
    prob_spam = p[1]
    bloquear = prob_spam > UMBRAL_BLOQUEO
    etiqueta = "SPAM" if r == 1 else "LEGÍTIMA"
    print(f"{desc}")
    print(f"  -> Predicción: {etiqueta}  |  Prob. Spam: {prob_spam:.2f}  |  Bloqueo automático (RF-04): {'SÍ' if bloquear else 'NO'}")
    print()

## 10. Exportar reglas de un árbol (referencia para la Fase 2 en Kotlin)

El Random Forest se reimplementará como lógica de reglas directas en Kotlin (no se necesita TensorFlow Lite). Esta salida ayuda a traducir manualmente los splits de un árbol individual a sentencias `if/else`.

In [ ]:
from sklearn.tree import export_text

arbol_ejemplo = modelo.estimators_[0]
reglas = export_text(arbol_ejemplo, feature_names=variables, max_depth=3)
print(reglas)

## 11. Guardar el modelo entrenado

In [ ]:
import joblib

joblib.dump(modelo, 'modelo_antispam_rf.joblib')
print("Modelo guardado en 'modelo_antispam_rf.joblib'")